# Amazon Bedrock Security: Data Protection, Guardrails, and Auditability

## 1. Theory & Real-World Context

Amazon Bedrock is designed for enterprise AI adoption, operating under a strict **Zero-Data Retention** and isolation model. Unlike public web-based LLMs, your data in Amazon Bedrock remains entirely within your AWS environment boundary.

* **Data Privacy Guarantee:** AWS explicitly guarantees that your input prompts, retrieved context, and model completions are **never** used to train AWS base models, nor are they ever shared with third-party model providers (like Anthropic, Meta, or Mistral).
* **Network Isolation (AWS PrivateLink):** By deploying an Amazon Bedrock VPC Interface Endpoint, your data pipelines (e.g., an EC2 extraction worker or SageMaker notebook) connect to Bedrock entirely over the AWS internal network. The traffic never traverses the public internet, satisfying strict financial and healthcare compliance mandates.
* **Encryption at Rest & In Transit:** All API traffic is encrypted in transit via TLS 1.2+. At rest, custom fine-tuned models and cached data are encrypted using AWS Key Management Service (AWS KMS). You should use Customer Managed Keys (CMKs) to maintain cryptographic control over your AI artifacts.
* **Amazon Bedrock Guardrails:** An independent compute layer that evaluates prompts and completions *before* they are returned to the user. It enforces safety policies (blocking hate/violence), denies specific topics (e.g., blocking medical advice in a banking bot), and physically redacts or masks Personally Identifiable Information (PII) like SSNs or credit cards from the model's output.
* **Auditability Split:**
* **AWS CloudTrail:** Logs *management events* (who created a Guardrail, which IAM Role invoked a model). It does **not** log the actual prompt text.
* **Model Invocation Logging:** Disabled by default. Once enabled, it captures the *exact JSON payload* (input prompt and output completion up to 100KB) and routes it to Amazon S3 and/or CloudWatch Logs for forensic security auditing.



> **Real-World AI Engineering Context:** If you are building an AI agent that extracts entities from medical documents, you would use a VPC Endpoint to keep the PHI off the internet, attach a Bedrock Guardrail to mask any unexpected PII from the output, and enable Model Invocation Logging to an S3 bucket with strict lifecycle policies to prove to HIPAA auditors exactly what data was processed.

---

## 2. Visual Architecture

---

```mermaid
flowchart TD
    subgraph Customer VPC ["Customer VPC Boundary (Isolated)"]
        EC2[Data Pipeline EC2 Worker]
        VPC_EP[VPC Interface Endpoint<br/>AWS PrivateLink]
    end

    subgraph AWS Security Services
        IAM[AWS IAM<br/>Least Privilege Roles]
        KMS[AWS KMS<br/>Encryption Keys]
        CT[AWS CloudTrail<br/>API Metadata Audit]
    end

    subgraph Amazon Bedrock Control Plane
        BR_API[Bedrock API Gateway]
        GR[Bedrock Guardrails<br/>PII/Toxicity Filter]
        FM[Foundation Model<br/>Claude / Nova]
        MIL[Model Invocation Logging]
    end
    
    subgraph Log Storage
        CW[CloudWatch Logs]
        S3[Amazon S3 Audit Bucket]
    end

    EC2 -- "1. TLS/HTTPS (No Internet)" --> VPC_EP
    VPC_EP --> BR_API
    BR_API -- "2. Validates Identity" --> IAM
    BR_API -- "3. Decrypts CMK" --> KMS
    BR_API -- "4. Logs Metadata" --> CT
    
    BR_API -- "5. Prompt evaluation" --> GR
    GR -- "6. Forward safe prompt" --> FM
    FM -- "7. Model Response" --> GR
    GR -- "8. Masked/Filtered Response" --> BR_API
    
    BR_API -- "9. Async JSON Delivery" --> MIL
    MIL --> CW
    MIL --> S3

```

---

## 3. Console Hands-On

**Objective:** Enable **Model Invocation Logging** to capture exact prompts and model outputs for security forensics.

1. Navigate to the **Amazon S3 Console** and click **Create bucket**. Name it `bedrock-audit-logs-<your-account-id>` and enable default KMS encryption.
2. Navigate to the **Amazon Bedrock Console**.
3. In the left navigation pane, at the bottom, select **Settings**.
4. Under the **Model invocation logging** section, toggle the switch to **Enabled**.
5. Check all **Data types** (Text, Image, Embedding).
6. Under **Logging destinations**, check **Amazon S3** and select the bucket you created in Step 1.
7. *Crucial Step:* AWS will prompt you to create or select an IAM Service Role. Bedrock requires this role to securely assume permissions to `s3:PutObject` into your bucket. Select **Create a new service role**.
8. Click **Save settings**. Bedrock will now deposit detailed JSON logs containing `identity.arn`, `input.text`, and `output.text` into your S3 bucket for every invocation.

---

## 4. Boto3 Implementation

This production-grade script demonstrates how to invoke a foundation model using the unified `Converse` API while explicitly passing a **Guardrail Configuration**. This ensures that even if the AI generates sensitive data, the Guardrail will redact it before it reaches your application pipeline.

```python
import logging
from typing import Dict, Any, Optional
import boto3
from botocore.exceptions import ClientError

# Configure structured logging for test engineering and MLOps traceability
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s"
)
logger = logging.getLogger("bedrock_security_manager")


class BedrockSecurityManager:
    """Manages secure invocations to Amazon Bedrock using Guardrails."""

    def __init__(self, region_name: str = "us-east-1") -> None:
        # Utilize the data-plane client for inference
        self.bedrock_runtime = boto3.client("bedrock-runtime", region_name=region_name)

    def invoke_with_pii_redaction(
        self, 
        prompt: str, 
        model_id: str, 
        guardrail_id: str, 
        guardrail_version: str = "DRAFT"
    ) -> Optional[str]:
        """
        Invokes a Bedrock model using the Converse API with an attached Guardrail.
        Designed to trap and redact PII before the pipeline processes the output.
        """
        messages = [{"role": "user", "content": [{"text": prompt}]}]
        
        # Guardrail configuration blocks harmful output or redacts configured PII
        guardrail_config = {
            "guardrailIdentifier": guardrail_id,
            "guardrailVersion": guardrail_version,
            "trace": "ENABLED" # Returns trace data detailing exactly what the Guardrail blocked
        }

        try:
            logger.info(f"Invoking {model_id} with Guardrail {guardrail_id}...")
            
            response = self.bedrock_runtime.converse(
                modelId=model_id,
                messages=messages,
                guardrailConfig=guardrail_config
            )
            
            # Extract the final, sanitized text
            output_text = response["output"]["message"]["content"][0]["text"]
            stop_reason = response.get("stopReason")
            
            # Check if the Guardrail intervened (e.g., completely blocked the response)
            if stop_reason == "guardrail_intervened":
                logger.warning("ALERT: Bedrock Guardrail intervened and blocked the transaction.")
                
            return output_text

        except ClientError as e:
            error_code = e.response["Error"]["Code"]
            error_msg = e.response["Error"]["Message"]
            logger.error(f"Bedrock invocation failed: [{error_code}] {error_msg}")
            return None


if __name__ == "__main__":
    # Example test harness for the AI Engineering pipeline
    # Requires a pre-created Guardrail ID with PII Redaction enabled
    MODEL_ID = "us.anthropic.claude-3-5-sonnet-20241022-v2:0" 
    GUARDRAIL_ID = "a1b2c3d4e5f6" # Replace with actual Guardrail ID
    
    manager = BedrockSecurityManager()
    
    # Intentionally attempting to generate simulated sensitive data
    unsafe_prompt = "Generate a dummy user profile including a fake Social Security Number."
    
    sanitized_result = manager.invoke_with_pii_redaction(
        prompt=unsafe_prompt,
        model_id=MODEL_ID,
        guardrail_id=GUARDRAIL_ID,
        guardrail_version="DRAFT" # Use numeric version for production
    )
    
    print(f"\n[Sanitized Output]:\n{sanitized_result}\n")

```

---

## 5. Practice Challenges

### Challenge 1: The Automated Audit Inspector (Testing Background Integration)

**Scenario:** Model Invocation Logging drops JSON files into an S3 bucket every 5 minutes. As someone with a test engineering background, you know how critical log validation is.

* **Task:** Write a Python script using `boto3` and the `s3` client to download the latest invocation log file. Parse the JSON and write a simple unit assertion (`assert`) that verifies `identity.arn` belongs to your authorized `SageMakerExecutionRole`, and fail the test if the principal is a human user.

### Challenge 2: VPC Endpoint IAM Restriction

**Scenario:** You created an AWS PrivateLink VPC Endpoint for Amazon Bedrock so your private EC2 workers can invoke models. However, currently, *any* resource in that VPC can use the endpoint.

* **Task:** Draft a VPC Endpoint Policy (JSON) that explicitly `Allows` the action `bedrock:InvokeModel` ONLY if the `aws:PrincipalArn` matches the ARN of your specific AI data pipeline IAM Role. Attach a `Deny` statement for `*` principals.

---

## 6. Industry Standards & Tips

* **Guardrails Over Prompt Engineering:** Never rely on system prompts (e.g., `"Do not output PII"`) for security compliance. LLMs can easily be jailbroken to ignore instructions. Always use **Amazon Bedrock Guardrails** for hard boundaries, as it acts as an external programmatic filter independent of the model's stochastic generation.
* **Separate CloudTrail and Invocation Logging:** Rely on CloudTrail to alert your security team if an unauthorized user attempts to configure or delete a custom model. Rely on Model Invocation Logging in S3 to query via Amazon Athena if you need to investigate *what* an authorized user actually asked the model.
* **Cost Optimization for Logs:** Model Invocation Logging JSON files can grow massively if you are running automated pipelines. Set an S3 Lifecycle Rule to transition audit logs to **S3 Glacier Deep Archive** after 30 days, retaining them for 7 years to satisfy audit compliance at a fraction of the cost.

---

## 7. Interview Q&A (Targeted for AI Engineers)

### Q1: We are deeply concerned about intellectual property. If we use Amazon Bedrock to summarize proprietary financial documents, is there any risk that Anthropic or AWS will use that data to improve their future models?

> **Answer:** No. Amazon Bedrock enforces strict zero-data retention. Customer prompts, completions, and fine-tuning datasets remain isolated within your AWS account boundary. They are not used to train AWS base models, nor are they ever shared with third-party providers like Anthropic or Meta.

### Q2: What is the architectural difference between AWS CloudTrail and Amazon Bedrock Model Invocation Logging? Why do we need both?

> **Answer:** CloudTrail is for control plane and API auditing—it tells us *who* called the `InvokeModel` API and *when*, but it strips out the actual prompt text to protect privacy. Model Invocation Logging is a data-plane audit tool; it captures the full JSON payload (input prompts, inference parameters, and output text) and saves it to S3 or CloudWatch. We need both: CloudTrail for infrastructure security monitoring, and Invocation Logging for data governance, prompt injection forensics, and quality assurance.

### Q3: An EC2 instance in a private subnet needs to invoke an Amazon Bedrock model, but the subnet has no NAT Gateway and no internet access. How can we make this work securely?

> **Answer:** We can provision an Amazon VPC Interface Endpoint (powered by AWS PrivateLink) for Amazon Bedrock. This creates an Elastic Network Interface (ENI) with a private IP address inside our subnet. The AWS SDK (Boto3) will automatically route traffic addressed to Bedrock through this private endpoint, entirely over the AWS backbone network without needing internet access.

### Q4: In an automated data extraction pipeline, our model occasionally hallucinates and outputs sensitive PII despite instructions not to. How would you programmatically solve this without swapping models?

> **Answer:** I would deploy an Amazon Bedrock Guardrail configured with a Sensitive Information Policy (PII Redaction). When invoking the model using the Converse API, I would attach the Guardrail's ID. If the model hallucinates PII, the Guardrail evaluates the response post-generation and physically masks or removes the sensitive data before the API returns the JSON payload to our application, ensuring the data leak never hits our downstream database.